# Analítica Empresarial Integrada — Laboratorio D14
## Semana 14 · IA generativa en la analítica empresarial
**TECSUP 2026-II · Docente: Pilar Rocío Sayán Mejía**

### Asistente analítico Text-to-SQL con evaluación de exactitud, alucinación y seguridad
**Datos reales:** Online Retail II — UCI Machine Learning Repository (ID 502) · **Motor:** DuckDB · **LLM:** el proveedor que defina el curso, conectado en la función `generate_sql()`.

**Pregunta de negocio:** ¿cómo construir un asistente que convierta preguntas empresariales en SQL útil, mida cuándo acierta y detecte cuándo inventa tablas, columnas o respuestas que no están respaldadas por los datos?

> El LLM es un componente de la solución, no un oráculo: su salida se valida, se ejecuta bajo control y se compara con una referencia. El objetivo no es una demostración que parezca funcionar, sino **medir** el desempeño y documentar las fallas.

## Identificación
- Equipo: Individual
- Sección: C28 6to A-L
- Integrante: Jhadir Abdel
- Repositorio: https://github.com/jhadir08/analitica-empresarial-integrada
- Proveedor y modelo de LLM utilizados: OpenAI · gpt-6-luna
- Uso de IA declarado: ChatGPT se utilizó como apoyo para comprender el código y organizar la redacción de respuestas e interpretaciones; los resultados numéricos corresponden a la ejecución de este notebook.
- Entorno de trabajo: Google Colab

## Objetivos y teoría
- **Text-to-SQL:** traducción de una pregunta en lenguaje natural a una consulta SQL sobre un esquema conocido.
- **Alucinación de esquema:** referencia a tablas, columnas o relaciones inexistentes. **Error semántico:** la consulta es válida pero no responde bien la pregunta.
- **Exactitud por ejecución:** compara el resultado del SQL generado con el de una consulta de referencia.
- **Seguridad:** el asistente solo puede producir `SELECT`; la protección real está en el validador y en una base de solo lectura con límite de filas, no en pedirle al modelo que sea cuidadoso.
- **Clave de acceso:** nunca se escribe en el notebook; se guarda en los *secretos* de Colab o en una variable de entorno.

In [23]:
# Instala en Colab las librerías que no vienen por defecto
%pip install -q "ucimlrepo>=0.0.7" duckdb openpyxl

In [24]:
import os
import re
import sys
from datetime import datetime, timezone
from io import BytesIO
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

import duckdb
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
# Parámetros del laboratorio
MAX_FILAS = 1000          # tope de filas que devuelve cualquier consulta generada
SEMILLA = 2026
VERSIONES = {"python": sys.version.split()[0], "pandas": pd.__version__, "duckdb": duckdb.__version__}
print(VERSIONES)

{'python': '3.13.18', 'pandas': '2.2.3', 'duckdb': '1.3.2'}


# 1. Datos reales y tabla `retail`
Online Retail II se distribuye como un libro de Excel con dos hojas (2009-2010 y 2010-2011). La celda siguiente intenta la API `ucimlrepo`, luego el ZIP oficial de UCI y, como último recurso, el archivo `online_retail_II.xlsx` en la carpeta de la sesión. El archivo supera el millón de filas: la carga puede tardar varios minutos. **No sustituya la fuente ni genere datos ficticios.**

In [25]:
try:
    retail_uci = fetch_ucirepo(id=502)
    df_raw = retail_uci.data.original.copy()
    fuente = "API ucimlrepo, UCI ID 502"
except Exception as exc:
    print("API no disponible:", type(exc).__name__)
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=180) as respuesta:
            with ZipFile(BytesIO(respuesta.read())) as z:
                nombre = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
                df_raw = pd.concat(pd.read_excel(BytesIO(z.read(nombre)), sheet_name=None).values(), ignore_index=True)
        fuente = "ZIP oficial UCI, ID 502"
    except Exception:
        local = Path("online_retail_II.xlsx")
        if not local.exists():
            raise FileNotFoundError("Descargue online_retail_II.xlsx desde UCI y súbalo a Colab, o solicite una copia a la docente.")
        df_raw = pd.concat(pd.read_excel(local, sheet_name=None).values(), ignore_index=True)
        fuente = "Archivo local descargado de UCI"
print("Fuente:", fuente, "· lectura UTC:", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"))
print("Filas originales:", len(df_raw), "· columnas:", list(df_raw.columns))

API no disponible: DatasetNotFoundError
Fuente: ZIP oficial UCI, ID 502 · lectura UTC: 2026-11-22 01:04:10
Filas originales: 1067371 · columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [26]:
# Nombres de columna únicos: el archivo puede traer Invoice o InvoiceNo, Price o UnitPrice, Customer ID o CustomerID
renombrar = {"Invoice": "InvoiceNo", "Price": "UnitPrice", "Customer ID": "CustomerID"}
df = df_raw.rename(columns={k: v for k, v in renombrar.items() if k in df_raw.columns}).copy()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
# Reglas de limpieza: ventas válidas, sin cancelaciones (facturas con prefijo C), con fecha, producto y país
cancelacion = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
valido = (~cancelacion) & df["InvoiceDate"].notna() & df["StockCode"].notna() & df["Country"].notna() & (df["Quantity"] > 0) & (df["UnitPrice"] > 0)
limpio = df.loc[valido].copy()
# Tabla analítica con los nombres de columna documentados; sales = quantity * unit_price
tabla = pd.DataFrame({
    "invoice_no": limpio["InvoiceNo"].astype(str),
    "invoice_date": limpio["InvoiceDate"],
    "stock_code": limpio["StockCode"].astype(str).str.strip(),
    "description": limpio["Description"].astype("string").str.strip(),
    "quantity": limpio["Quantity"].astype("int64"),
    "unit_price": limpio["UnitPrice"].astype(float),
    "customer_id": limpio["CustomerID"].astype("Int64"),
    "country": limpio["Country"].astype(str).str.strip(),
})
tabla["sales"] = tabla["quantity"] * tabla["unit_price"]
print("Cancelaciones retiradas:", int(cancelacion.sum()), "· filas válidas:", len(tabla))
print("Periodo:", tabla["invoice_date"].min(), "→", tabla["invoice_date"].max())
display(tabla.head())

Cancelaciones retiradas: 19494 · filas válidas: 1041670
Periodo: 2009-12-01 07:45:00 → 2011-12-09 12:50:00


,invoice_no,invoice_date,stock_code,description,quantity,unit_price,customer_id,country,sales
0,489434,2009-12-01 07:45:00,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,13085,United Kingdom,83.4
1,489434,2009-12-01 07:45:00,79323P,PINK CHERRY LIGHTS,12,6.75,13085,United Kingdom,81.0
2,489434,2009-12-01 07:45:00,79323W,WHITE CHERRY LIGHTS,12,6.75,13085,United Kingdom,81.0
3,489434,2009-12-01 07:45:00,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,13085,United Kingdom,100.8
4,489434,2009-12-01 07:45:00,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,13085,United Kingdom,30.0


In [30]:
RUTA_BD = "retail.duckdb"

# Si esta celda ya se ejecutó, cerrar la conexión de solo lectura anterior
if "con" in globals():
    con.close()

# Cerrar también una conexión de escritura anterior, si quedó abierta
if "con_admin" in globals():
    con_admin.close()

# Crear o reemplazar la tabla con los datos actuales de pandas
con_admin = duckdb.connect(RUTA_BD)
try:
    con_admin.register("tabla_pandas", tabla)
    con_admin.execute(
        "CREATE OR REPLACE TABLE retail AS SELECT * FROM tabla_pandas"
    )
finally:
    con_admin.close()

# El asistente utilizará esta conexión de solo lectura
con = duckdb.connect(RUTA_BD, read_only=True)

print(con.execute("""
    SELECT COUNT(*) AS filas,
           MIN(invoice_date) AS desde,
           MAX(invoice_date) AS hasta
    FROM retail
""").df())

     filas               desde               hasta
0  1041670 2009-12-01 07:45:00 2011-12-09 12:50:00


# 2. Diccionario de datos y contexto para el LLM
El asistente recibe **únicamente** este contexto. Los nombres ambiguos o las definiciones ausentes aumentan los errores.

In [31]:
diccionario = pd.DataFrame([
    ["invoice_no", "texto", "Número de factura; cada factura puede tener varias líneas."],
    ["invoice_date", "fecha y hora", "Momento de la factura."],
    ["stock_code", "texto", "Código del producto; úselo como identificador del producto."],
    ["description", "texto", "Descripción del producto; puede variar para un mismo código."],
    ["quantity", "entero", "Unidades vendidas en la línea (siempre mayor que cero tras la limpieza)."],
    ["unit_price", "decimal", "Precio unitario en libras esterlinas."],
    ["customer_id", "entero", "Identificador del cliente; tiene valores faltantes."],
    ["country", "texto", "País del cliente, en inglés (por ejemplo, Germany, France)."],
    ["sales", "decimal", "Venta de la línea: quantity * unit_price."],
], columns=["columna", "tipo", "definicion"])
display(diccionario)
SYSTEM_CONTEXT = '''
Tabla disponible: retail
Columnas: invoice_no, invoice_date, stock_code, description,
quantity, unit_price, customer_id, country, sales.
Reglas: solo SELECT; usar sales = quantity * unit_price;
no inventar tablas ni columnas.
'''
# El contexto completo incluye el diccionario de datos, el formato de salida y la política de seguridad
CONTEXTO_LLM = SYSTEM_CONTEXT + "\nDiccionario de datos:\n" + "\n".join(f"- {r.columna} ({r.tipo}): {r.definicion}" for r in diccionario.itertuples()) + "\nSalida: responda únicamente con una consulta SQL SELECT de DuckDB, sin explicaciones ni formato de código."
print(CONTEXTO_LLM)

,columna,tipo,definicion
0,invoice_no,texto,Número de factura; cada factura puede tener varias líneas.
1,invoice_date,fecha y hora,Momento de la factura.
2,stock_code,texto,Código del producto; úselo como identificador del producto.
3,description,texto,Descripción del producto; puede variar para un mismo código.
4,quantity,entero,Unidades vendidas en la línea (siempre mayor que cero tras la limpieza).
5,unit_price,decimal,Precio unitario en libras esterlinas.
6,customer_id,entero,Identificador del cliente; tiene valores faltantes.
7,country,texto,"País del cliente, en inglés (por ejemplo, Germany, France)."
8,sales,decimal,Venta de la línea: quantity * unit_price.



Tabla disponible: retail
Columnas: invoice_no, invoice_date, stock_code, description,
quantity, unit_price, customer_id, country, sales.
Reglas: solo SELECT; usar sales = quantity * unit_price;
no inventar tablas ni columnas.

Diccionario de datos:
- invoice_no (texto): Número de factura; cada factura puede tener varias líneas.
- invoice_date (fecha y hora): Momento de la factura.
- stock_code (texto): Código del producto; úselo como identificador del producto.
- description (texto): Descripción del producto; puede variar para un mismo código.
- quantity (entero): Unidades vendidas en la línea (siempre mayor que cero tras la limpieza).
- unit_price (decimal): Precio unitario en libras esterlinas.
- customer_id (entero): Identificador del cliente; tiene valores faltantes.
- country (texto): País del cliente, en inglés (por ejemplo, Germany, France).
- sales (decimal): Venta de la línea: quantity * unit_price.
Salida: responda únicamente con una consulta SQL SELECT de DuckDB, sin explic

## Pregunta 1
**Respuesta:** La sintaxis solo confirma que DuckDB puede interpretar la consulta; no que mida el concepto empresarial correcto. Sin definiciones de granularidad, unidades y reglas, el modelo puede sumar líneas cuando se solicitan facturas distintas, confundir precio con ventas o aplicar filtros equivocados. El resultado se ejecuta, pero responde otra pregunta.

## Pregunta 2
**Respuesta:** Documentar el grano de cada fila, tipos y significado de columnas, claves y relaciones, tratamiento de nulos y cancelaciones, periodo y filtros válidos, unidades y moneda, y fórmulas de negocio. En este caso debe aclararse que una factura tiene varias líneas y que sales es quantity por unit_price en libras esterlinas; también cuándo contar números de factura distintos.

# 3. Conexión al LLM: la función `generate_sql()`
`generate_sql(pregunta)` recibe una pregunta y devuelve **la respuesta cruda del modelo**. Para conectarla, el equipo completa `llamar_llm()` con el proveedor que defina el curso. Reglas:

- La **clave** se lee de los *secretos* de Colab (`LLM_API_KEY`) o de una variable de entorno; **no se escribe en el notebook ni se imprime**.
- Use temperatura 0 (o la más baja disponible) para que los resultados sean repetibles, y registre proveedor y modelo en la identificación.
- El modelo recibe `CONTEXTO_LLM` como instrucción de sistema y la pregunta como mensaje del usuario.

Ejemplos orientativos (verifique la documentación vigente del proveedor; los nombres de modelo cambian):

```python
# Anthropic
import anthropic
cliente = anthropic.Anthropic(api_key=CLAVE)
r = cliente.messages.create(model=MODELO, max_tokens=400, temperature=0, system=sistema,
                            messages=[{"role": "user", "content": pregunta}])
texto = r.content[0].text

# OpenAI
from openai import OpenAI
cliente = OpenAI(api_key=CLAVE)
r = cliente.chat.completions.create(model=MODELO, temperature=0,
        messages=[{"role": "system", "content": sistema}, {"role": "user", "content": pregunta}])
texto = r.choices[0].message.content

# Google Gemini
from google import genai
cliente = genai.Client(api_key=CLAVE)
r = cliente.models.generate_content(model=MODELO, contents=pregunta,
        config={"system_instruction": sistema, "temperature": 0})
texto = r.text
```

In [32]:
# Conexión a OpenAI para la batería de Text-to-SQL
import os
import requests

PROVEEDOR = "openai"
MODELO = "gpt-6-luna"


def obtener_clave():
    try:
        from google.colab import userdata
    except ImportError:
        clave = os.environ.get("LLM_API_KEY", "").strip()
    else:
        try:
            clave = (userdata.get("LLM_API_KEY") or "").strip()
        except userdata.SecretNotFoundError:
            clave = os.environ.get("LLM_API_KEY", "").strip()
        except userdata.NotebookAccessError:
            raise RuntimeError("Habilita el acceso del notebook al secreto LLM_API_KEY en Colab.") from None
    if not clave:
        raise RuntimeError("Falta el secreto LLM_API_KEY con la clave de OpenAI.")
    return clave


def llamar_llm(sistema, pregunta):
    clave = obtener_clave()
    try:
        respuesta = requests.post(
            "https://api.openai.com/v1/responses",
            headers={"Authorization": f"Bearer {clave}"},
            json={
                "model": MODELO,
                "instructions": sistema,
                "input": pregunta,
                "reasoning": {"effort": "none"},
                "max_output_tokens": 400,
                "store": False,
            },
            timeout=45,
        )
    except requests.RequestException as exc:
        raise RuntimeError(f"No se pudo conectar con OpenAI ({type(exc).__name__}).") from None
    if not respuesta.ok:
        try:
            codigo = (respuesta.json().get("error") or {}).get("code")
        except ValueError:
            codigo = None
        detalle = f" ({codigo})" if codigo in {"invalid_api_key", "insufficient_quota", "rate_limit_exceeded", "model_not_found"} else ""
        raise RuntimeError(f"OpenAI devolvió HTTP {respuesta.status_code}{detalle} para {MODELO}.")
    datos = respuesta.json()
    if datos.get("status") != "completed":
        raise RuntimeError("OpenAI no completó la generación del SQL.")
    texto = "".join(
        parte.get("text", "")
        for elemento in datos.get("output", []) if elemento.get("type") == "message"
        for parte in elemento.get("content", []) if parte.get("type") == "output_text"
    )
    if not texto.strip():
        raise RuntimeError("OpenAI respondió sin texto SQL.")
    return texto


def generate_sql(pregunta):
    return llamar_llm(CONTEXTO_LLM, pregunta)

# 4. Validación y ejecución bajo control
Cada consulta pasa por `validate_sql()` **antes** de llegar a DuckDB. Después se ejecuta con `run_sql()` sobre la conexión de solo lectura y con un tope de filas.

In [33]:
PALABRAS_PROHIBIDAS = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "COPY", "ATTACH", "DETACH", "PRAGMA",
                       "EXPORT", "IMPORT", "INSTALL", "LOAD", "CALL", "SET", "TRUNCATE", "REPLACE", "MERGE", "READ_CSV",
                       "READ_PARQUET", "READ_JSON"]


def limpiar_sql(texto):
    # Quita las marcas de formato de código (```sql ... ```) y el punto y coma final
    t = texto.strip()
    t = re.sub(r"^```(?:sql)?\s*", "", t, flags=re.I)
    t = re.sub(r"\s*```$", "", t)
    return t.strip().rstrip(";").strip()


def validate_sql(sql):
    # Devuelve (permitida, motivo). Regla: una sola sentencia que empiece con SELECT o WITH y sin palabras prohibidas
    s = sql.strip()
    if not s:
        return False, "consulta vacía"
    sin_cadenas = re.sub(r"'[^']*'", "''", s)
    if ";" in sin_cadenas.rstrip().rstrip(";"):
        return False, "más de una sentencia"
    if not re.match(r"^(SELECT|WITH)\b", s, flags=re.I):
        return False, "no empieza con SELECT"
    for palabra in PALABRAS_PROHIBIDAS:
        if re.search(r"\b" + palabra + r"\b", sin_cadenas, flags=re.I):
            return False, f"palabra prohibida: {palabra}"
    return True, "permitida"


def run_sql(sql, max_filas=MAX_FILAS):
    # Ejecuta la consulta en la conexión de solo lectura y limita las filas devueltas
    return con.execute(f"SELECT * FROM ({sql}) AS consulta LIMIT {max_filas}").df()

In [34]:
# Pruebas del validador: no necesitan LLM
casos_prueba = [
    ("SELECT COUNT(*) FROM retail", True),
    ("WITH t AS (SELECT country FROM retail) SELECT COUNT(*) FROM t", True),
    ("DELETE FROM retail", False),
    ("DROP TABLE retail", False),
    ("SELECT * FROM retail; DROP TABLE retail", False),
    ("INSERT INTO retail VALUES (1)", False),
    ("SELECT * FROM read_csv('datos.csv')", False),
    ("SELECT description FROM retail WHERE description = 'DELETE ME'", True),
]
for sql, esperado in casos_prueba:
    permitida, motivo = validate_sql(sql)
    assert permitida == esperado, (sql, motivo)
print("El validador pasó las", len(casos_prueba), "pruebas.")
# La conexión de solo lectura rechaza escrituras aunque el validador fallara
try:
    con.execute("DELETE FROM retail")
except Exception as exc:
    print("Conexión de solo lectura:", type(exc).__name__)

El validador pasó las 8 pruebas.
Conexión de solo lectura: InvalidInputException


## Pregunta 3
**Respuesta:** Un prompt es una instrucción, no un control que el modelo esté obligado a respetar. El LLM puede equivocarse, ignorarlo o recibir una pregunta maliciosa; por eso una respuesta peligrosa todavía puede generarse. La aplicación debe validar el SQL fuera del modelo y ejecutarlo con permisos mínimos y una conexión de solo lectura.

## Pregunta 4
**Respuesta:** La seguridad del prompt orienta al modelo para que responda solo con consultas apropiadas, pero no garantiza el cumplimiento. La seguridad de ejecución aplica barreras técnicas independientes: aceptar únicamente una sentencia SELECT/CTE, rechazar operaciones y funciones peligrosas, usar una conexión de solo lectura y limitar filas y recursos.

# 5. Batería de evaluación
Diez preguntas empresariales con su consulta de referencia. Para cada caso se registra el SQL generado, el estado de seguridad, la ejecución, la coincidencia del resultado y el tipo de falla. **Tipos de falla:** `esquema` (tabla o columna inexistente), `sintaxis/ejecución`, `semántica` (ejecuta pero el resultado difiere), `seguridad` (bloqueada) y `formato` (la respuesta trajo texto o marcas de código además del SQL).

In [35]:
BATERIA = [
    (1, "¿Cuántas facturas distintas hay en total?",
     "SELECT COUNT(DISTINCT invoice_no) FROM retail"),
    (2, "¿Cuál es la venta total (sales) de toda la tabla?",
     "SELECT SUM(sales) FROM retail"),
    (3, "¿Cuáles son los 5 países con mayores ventas totales?",
     "SELECT country, SUM(sales) AS ventas FROM retail GROUP BY country ORDER BY ventas DESC LIMIT 5"),
    (4, "¿Cuántos clientes distintos (customer_id) compraron desde Germany?",
     "SELECT COUNT(DISTINCT customer_id) FROM retail WHERE country = 'Germany'"),
    (5, "¿Cuál es el ticket promedio por factura, entendido como la venta total dividida entre el número de facturas distintas?",
     "SELECT SUM(sales) / COUNT(DISTINCT invoice_no) FROM retail"),
    (6, "¿Cuáles son los 3 productos (stock_code) con más unidades vendidas?",
     "SELECT stock_code, SUM(quantity) AS unidades FROM retail GROUP BY stock_code ORDER BY unidades DESC LIMIT 3"),
    (7, "¿Qué mes de 2011 tuvo las mayores ventas? Indique el mes como año-mes.",
     "SELECT strftime(invoice_date, '%Y-%m') AS mes, SUM(sales) AS ventas FROM retail WHERE year(invoice_date) = 2011 GROUP BY mes ORDER BY ventas DESC LIMIT 1"),
    (8, "¿Cuántas líneas de factura tienen una venta (sales) mayor que 500?",
     "SELECT COUNT(*) FROM retail WHERE sales > 500"),
    (9, "¿Cuál es el precio unitario promedio de los productos vendidos en France?",
     "SELECT AVG(unit_price) FROM retail WHERE country = 'France'"),
    (10, "¿Cuántos clientes (customer_id) compraron en más de 5 facturas distintas?",
     "SELECT COUNT(*) FROM (SELECT customer_id FROM retail WHERE customer_id IS NOT NULL GROUP BY customer_id HAVING COUNT(DISTINCT invoice_no) > 5)"),
]
referencias = {n: run_sql(sql_ref) for n, _, sql_ref in BATERIA}
for n, pregunta, _ in BATERIA:
    print(n, pregunta, "→ referencia:", referencias[n].shape)

1 ¿Cuántas facturas distintas hay en total? → referencia: (1, 1)
2 ¿Cuál es la venta total (sales) de toda la tabla? → referencia: (1, 1)
3 ¿Cuáles son los 5 países con mayores ventas totales? → referencia: (5, 2)
4 ¿Cuántos clientes distintos (customer_id) compraron desde Germany? → referencia: (1, 1)
5 ¿Cuál es el ticket promedio por factura, entendido como la venta total dividida entre el número de facturas distintas? → referencia: (1, 1)
6 ¿Cuáles son los 3 productos (stock_code) con más unidades vendidas? → referencia: (3, 2)
7 ¿Qué mes de 2011 tuvo las mayores ventas? Indique el mes como año-mes. → referencia: (1, 2)
8 ¿Cuántas líneas de factura tienen una venta (sales) mayor que 500? → referencia: (1, 1)
9 ¿Cuál es el precio unitario promedio de los productos vendidos en France? → referencia: (1, 1)
10 ¿Cuántos clientes (customer_id) compraron en más de 5 facturas distintas? → referencia: (1, 1)


In [36]:
def valores(df_resultado, decimales=2):
    # Resultado como lista ordenada de filas redondeadas: se ignoran los nombres de columna
    filas = []
    for fila in df_resultado.itertuples(index=False):
        filas.append(tuple(round(float(v), decimales) if isinstance(v, (int, float, np.integer, np.floating)) and not pd.isna(v) else str(v) for v in fila))
    return sorted(filas, key=str)


def mismo_resultado(df_a, df_b):
    return df_a.shape == df_b.shape and valores(df_a) == valores(df_b)


def evaluar(n, pregunta):
    # Una pregunta: genera, valida, ejecuta, compara y clasifica la falla
    registro = {"n": n, "pregunta": pregunta, "respuesta_cruda": None, "sql": None, "seguridad": None,
                "ejecuta": False, "coincide": False, "tipo_falla": None, "detalle": ""}
    try:
        cruda = generate_sql(pregunta)
    except Exception as exc:
        registro.update(tipo_falla="sin respuesta", detalle=f"{type(exc).__name__}: {exc}"[:200])
        return registro
    registro["respuesta_cruda"] = cruda
    sql = limpiar_sql(cruda)
    registro["sql"] = sql
    formato_ok = cruda.strip().rstrip(";").strip() == sql
    permitida, motivo = validate_sql(sql)
    registro["seguridad"] = motivo
    if not permitida:
        registro.update(tipo_falla="seguridad", detalle=motivo)
        return registro
    try:
        resultado = run_sql(sql)
        registro["ejecuta"] = True
    except Exception as exc:
        mensaje = str(exc)
        clase = "esquema" if re.search(r"Catalog Error|Binder Error|does not exist|not found in FROM|Referenced column", mensaje, re.I) else "sintaxis/ejecución"
        registro.update(tipo_falla=clase, detalle=mensaje.splitlines()[0][:200])
        return registro
    registro["coincide"] = mismo_resultado(resultado, referencias[n])
    if not registro["coincide"]:
        registro.update(tipo_falla="semántica", detalle="el resultado difiere de la referencia")
    elif not formato_ok:
        registro.update(tipo_falla="formato", detalle="la respuesta incluyó texto o marcas de código además del SQL")
    return registro

In [37]:
# Se evalúa toda la batería; si el LLM no está conectado, cada caso queda como «sin respuesta»
bateria = pd.DataFrame([evaluar(n, pregunta) for n, pregunta, _ in BATERIA])
display(bateria[["n", "seguridad", "ejecuta", "coincide", "tipo_falla", "detalle"]])

,n,seguridad,ejecuta,coincide,tipo_falla,detalle
0,1,permitida,True,True,None,
1,2,permitida,True,True,None,
2,3,permitida,True,True,None,
3,4,permitida,True,True,None,
4,5,permitida,True,True,None,
5,6,permitida,True,True,None,
6,7,permitida,True,False,semántica,el resultado difiere de la referencia
7,8,permitida,True,True,None,
8,9,permitida,True,True,None,
9,10,permitida,True,True,None,


In [38]:
# Indicadores de la batería
n_preguntas = len(bateria)
aciertos = int(bateria.coincide.sum())
indicadores = pd.DataFrame([
    ["Exactitud de ejecución (%)", 100 * aciertos / n_preguntas],
    ["Tasa de alucinación de esquema (%)", 100 * (bateria.tipo_falla == "esquema").sum() / n_preguntas],
    ["Tasa de error semántico (%)", 100 * (bateria.tipo_falla == "semántica").sum() / n_preguntas],
    ["Tasa de bloqueo por seguridad (%)", 100 * (bateria.tipo_falla == "seguridad").sum() / n_preguntas],
    ["Tasa de error de sintaxis o ejecución (%)", 100 * (bateria.tipo_falla == "sintaxis/ejecución").sum() / n_preguntas],
    ["Respuestas con formato incorrecto (%)", 100 * (bateria.tipo_falla == "formato").sum() / n_preguntas],
], columns=["indicador", "valor"])
display(indicadores.round(1))
if (bateria.tipo_falla == "sin respuesta").any():
    print("Atención: hay preguntas sin respuesta del LLM; conecte generate_sql() antes de interpretar los indicadores.")

,indicador,valor
0,Exactitud de ejecución (%),90.0
1,Tasa de alucinación de esquema (%),0.0
2,Tasa de error semántico (%),10.0
3,Tasa de bloqueo por seguridad (%),0.0
4,Tasa de error de sintaxis o ejecución (%),0.0
5,Respuestas con formato incorrecto (%),0.0


## Pregunta 5
**Respuesta:** Se evaluaron 10 preguntas y 9 coincidieron con las consultas de referencia: exactitud de ejecución de 90%. La única falla fue semántica (10%), en la pregunta 7. No hubo alucinaciones de esquema, errores de sintaxis o ejecución, bloqueos de seguridad ni errores de formato.

## Pregunta 6
**Respuesta:** No se observó alucinación de esquema en la batería: las 10 consultas usaron tablas y columnas existentes, por lo que la tasa fue 0%. El asistente coincidió con las referencias en 9 de 10 preguntas (90%). La única falla fue semántica en la pregunta 7: el SQL se ejecutó, pero el mes de mayores ventas de 2011 no coincidió con la referencia. El diccionario de datos y las reglas de negocio reducen el riesgo de inventar elementos del esquema, aunque no garantizan que la consulta responda correctamente la intención empresarial.

# 6. Análisis de errores y mitigación
Clasifique cada falla como `esquema`, `sintaxis/ejecución`, `semántica`, `seguridad` o `formato` y proponga una mitigación **verificable**: algo que pueda volver a medirse con la misma batería.

In [41]:
# Detalle dinámico de las fallas semánticas observadas (no vuelve a llamar al LLM).
mitigaciones = {
    7: (
        "Aclarar que se busca el mes calendario de 2011 con mayor SUM(sales), agrupado por año-mes.",
        "Repetir la pregunta y verificar que mes y ventas coincidan con la consulta de referencia.",
    )
}
fallas = bateria.loc[bateria["tipo_falla"].eq("semántica"), ["n", "pregunta", "tipo_falla", "detalle"]].copy()
fallas["mitigacion_propuesta"] = fallas["n"].map(lambda n: mitigaciones.get(n, ("Revisar la definición de negocio y el grano solicitado.", None))[0])
fallas["como_se_verifica"] = fallas["n"].map(lambda n: mitigaciones.get(n, (None, "Comparar nuevamente con la consulta de referencia."))[1])
print("Fallas semánticas observadas en la batería:", len(fallas))
display(fallas)

Fallas semánticas observadas en la batería: 1


,n,pregunta,tipo_falla,detalle,mitigacion_propuesta,como_se_verifica
6,7,¿Qué mes de 2011 tuvo las mayores ventas? Indique el mes como año-mes.,semántica,el resultado difiere de la referencia,"Aclarar que se busca el mes calendario de 2011 con mayor SUM(sales), agrupado por año-mes.",Repetir la pregunta y verificar que mes y ventas coincidan con la consulta de referencia.


## Pregunta 7
**Respuesta:** El error semántico puede ser más riesgoso que un error de ejecución porque la consulta termina y produce un resultado plausible, aunque no responda correctamente la pregunta. Un error de ejecución suele ser visible y puede corregirse antes de usar el dato. En esta batería hubo una falla semántica (pregunta 7 de 10); por ello, incluso con 90% de coincidencia, se deben revisar las métricas importantes frente a una referencia.

## Pregunta 8
**Respuesta:** Mantendría conexión de solo lectura, validación de una única consulta SELECT/CTE y funciones permitidas, límites de filas y tiempo, registro de consultas, revisión humana para métricas importantes y pruebas periódicas contra referencias. Para producción reforzaría el validador con análisis sintáctico de SQL y controles de acceso; una lista de palabras prohibidas por expresiones regulares no es una garantía completa.

# 7. Cuándo NO usar un LLM
Evalúe si el asistente debería responder, pedir aclaración o derivar la consulta a un analista humano.

## Pregunta 9
**Respuesta:** «¿Qué país vende mejor?» requiere aclaración: “mejor” podría significar mayores ventas en libras, más unidades o más facturas, y falta indicar el periodo. El asistente debe preguntar qué métrica y periodo se desea antes de generar SQL; si el usuario no aclara, debe abstenerse y derivar la consulta.

## Pregunta 10
**Respuesta:** Recomendaría RAG para preguntas que requieren buscar y resumir contenido no estructurado —por ejemplo, políticas, manuales o contratos— y mostrar los pasajes que respaldan la respuesta. Text-to-SQL corresponde a preguntas de cálculo sobre tablas estructuradas. Si la respuesta depende tanto de cifras como de documentos, se puede combinar recuperación documental con consultas SQL verificables.

# Opcional. Llevar el asistente a una aplicación con chat (Streamlit) y trabajarlo en Cursor
Esta sección **no cambia** la evaluación del laboratorio: puede completarse en Colab, como hasta aquí, o llevarse a una aplicación que se ejecuta en **Cursor** (o en cualquier editor con Python) y se publica en **Streamlit Community Cloud**.

**Qué hace la aplicación:** una ventana de chat donde la persona escribe una pregunta de negocio. El asistente muestra el SQL que generó el modelo, el estado de la validación de seguridad y la tabla de resultados. Reutiliza exactamente la misma lógica de este notebook: `generate_sql()` (aquí `llamar_llm()`), `validate_sql()` y `run_sql()` sobre una base de **solo lectura**.

**Qué modelo usar**
- **Ollama** (gratuito): en Colab se ejecuta temporalmente en la sesión; en Cursor se instala en el computador. Solo sirve para ejecutar la aplicación en su propio equipo: **no corre en Streamlit Community Cloud**.
- **API de un proveedor** (Anthropic, OpenAI o Gemini): necesaria para la aplicación publicada. Cada persona usa su propia clave.

**Reglas de seguridad**
- La clave **nunca** se escribe en el código ni se sube a GitHub: va en los *secretos* de Streamlit o en una variable de entorno.
- La aplicación publicada tiene un tope de 20 preguntas por sesión y admite una contraseña (`APP_PASSWORD`) para evitar que otras personas gasten su crédito.

In [40]:
APP_PY = r'''# -*- coding: utf-8 -*-
"""Aplicación de chat del asistente text-to-SQL (Laboratorio D14). Se ejecuta con: streamlit run app.py"""
import os
import re

import duckdb
import pandas as pd
import streamlit as st

RUTA_BD = os.environ.get("RUTA_BD", "retail_app.duckdb")
MAX_FILAS = 1000
MAX_PREGUNTAS = 20  # tope de preguntas por sesión para cuidar el crédito de la clave

PALABRAS_PROHIBIDAS = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "COPY", "ATTACH", "DETACH", "PRAGMA",
                       "EXPORT", "IMPORT", "INSTALL", "LOAD", "CALL", "SET", "TRUNCATE", "REPLACE", "MERGE", "READ_CSV",
                       "READ_PARQUET", "READ_JSON"]

CONTEXTO_LLM = """Tabla disponible: retail
Columnas: invoice_no, invoice_date, stock_code, description, quantity, unit_price, customer_id, country, sales.
Reglas: solo SELECT; usar sales = quantity * unit_price; no inventar tablas ni columnas.
Diccionario de datos:
- invoice_no (texto): número de factura; cada factura puede tener varias líneas.
- invoice_date (fecha y hora): momento de la factura.
- stock_code (texto): código del producto; úselo como identificador del producto.
- description (texto): descripción del producto; puede variar para un mismo código.
- quantity (entero): unidades vendidas en la línea (siempre mayor que cero tras la limpieza).
- unit_price (decimal): precio unitario en libras esterlinas.
- customer_id (entero): identificador del cliente; tiene valores faltantes.
- country (texto): país del cliente, en inglés (por ejemplo, Germany, France).
- sales (decimal): venta de la línea, quantity * unit_price.
Salida: responda únicamente con una consulta SQL SELECT de DuckDB, sin explicaciones ni formato de código."""


def valor(nombre, defecto=None):
    # Busca primero en los secretos de Streamlit y luego en las variables de entorno
    try:
        return st.secrets[nombre]
    except Exception:
        return os.environ.get(nombre, defecto)


PROVEEDOR = valor("LLM_PROVEEDOR", "ollama")   # "ollama", "anthropic", "openai" o "gemini"
MODELO = valor("LLM_MODELO", "llama3.2")        # verifique el nombre vigente en la documentación del proveedor
CLAVE = valor("LLM_API_KEY")


def llamar_llm(sistema, pregunta):
    if PROVEEDOR == "ollama":
        import requests
        r = requests.post("http://localhost:11434/api/chat", timeout=120, json={
            "model": MODELO, "stream": False, "options": {"temperature": 0},
            "messages": [{"role": "system", "content": sistema}, {"role": "user", "content": pregunta}]})
        r.raise_for_status()
        return r.json()["message"]["content"]
    if not CLAVE:
        raise RuntimeError("Falta LLM_API_KEY en los secretos o en las variables de entorno.")
    if PROVEEDOR == "anthropic":
        import anthropic
        r = anthropic.Anthropic(api_key=CLAVE).messages.create(
            model=MODELO, max_tokens=400, temperature=0, system=sistema,
            messages=[{"role": "user", "content": pregunta}])
        return r.content[0].text
    if PROVEEDOR == "openai":
        from openai import OpenAI
        r = OpenAI(api_key=CLAVE).chat.completions.create(
            model=MODELO, temperature=0,
            messages=[{"role": "system", "content": sistema}, {"role": "user", "content": pregunta}])
        return r.choices[0].message.content
    if PROVEEDOR == "gemini":
        from google import genai
        r = genai.Client(api_key=CLAVE).models.generate_content(
            model=MODELO, contents=pregunta, config={"system_instruction": sistema, "temperature": 0})
        return r.text
    raise RuntimeError(f"Proveedor no reconocido: {PROVEEDOR}")


def limpiar_sql(texto):
    t = texto.strip()
    t = re.sub(r"^```(?:sql)?\s*", "", t, flags=re.I)
    t = re.sub(r"\s*```$", "", t)
    return t.strip().rstrip(";").strip()


def validate_sql(sql):
    s = sql.strip()
    if not s:
        return False, "consulta vacía"
    sin_cadenas = re.sub(r"'[^']*'", "''", s)
    if ";" in sin_cadenas.rstrip().rstrip(";"):
        return False, "más de una sentencia"
    if not re.match(r"^(SELECT|WITH)\b", s, flags=re.I):
        return False, "no empieza con SELECT"
    for palabra in PALABRAS_PROHIBIDAS:
        if re.search(r"\b" + palabra + r"\b", sin_cadenas, flags=re.I):
            return False, f"palabra prohibida: {palabra}"
    return True, "permitida"


@st.cache_resource
def conectar():
    # Conexión de solo lectura: el asistente no puede modificar la base aunque el validador fallara
    return duckdb.connect(RUTA_BD, read_only=True)


def run_sql(sql):
    return conectar().execute(f"SELECT * FROM ({sql}) AS consulta LIMIT {MAX_FILAS}").df()


st.set_page_config(page_title="Asistente analítico", page_icon="💬")
st.title("Asistente analítico text-to-SQL")
st.caption("Pregunte sobre las ventas de Online Retail II (tabla retail). El asistente genera una consulta SELECT, "
           "se valida antes de ejecutarla y puede equivocarse: revise siempre el SQL y el resultado.")

clave_app = valor("APP_PASSWORD")
if clave_app:
    if st.sidebar.text_input("Contraseña de la aplicación", type="password") != clave_app:
        st.info("Escriba la contraseña en la barra lateral para usar el asistente.")
        st.stop()

if "historial" not in st.session_state:
    st.session_state.historial = []
    st.session_state.preguntas = 0

for m in st.session_state.historial:
    with st.chat_message(m["rol"]):
        st.markdown(m["texto"])
        if m.get("sql"):
            st.code(m["sql"], language="sql")
        if m.get("tabla") is not None:
            st.dataframe(m["tabla"])

pregunta = st.chat_input("Escriba una pregunta de negocio sobre las ventas")
if pregunta:
    st.session_state.historial.append({"rol": "user", "texto": pregunta})
    with st.chat_message("user"):
        st.markdown(pregunta)
    respuesta = {"rol": "assistant", "texto": "", "sql": None, "tabla": None}
    if st.session_state.preguntas >= MAX_PREGUNTAS:
        respuesta["texto"] = "Se alcanzó el máximo de preguntas de esta sesión."
    else:
        st.session_state.preguntas += 1
        try:
            sql = limpiar_sql(llamar_llm(CONTEXTO_LLM, pregunta))
            respuesta["sql"] = sql
            permitida, motivo = validate_sql(sql)
            if not permitida:
                respuesta["texto"] = f"Consulta bloqueada por seguridad: {motivo}."
            else:
                try:
                    respuesta["tabla"] = run_sql(sql)
                    respuesta["texto"] = f"Consulta permitida. Filas devueltas: {len(respuesta['tabla'])}."
                except Exception as exc:
                    respuesta["texto"] = f"La consulta no se pudo ejecutar ({type(exc).__name__}): {exc}"
        except Exception as exc:
            respuesta["texto"] = f"No se pudo obtener respuesta del modelo: {exc}"
    st.session_state.historial.append(respuesta)
    with st.chat_message("assistant"):
        st.markdown(respuesta["texto"])
        if respuesta["sql"]:
            st.code(respuesta["sql"], language="sql")
        if respuesta["tabla"] is not None:
            st.dataframe(respuesta["tabla"])
'''

from pathlib import Path

Path("app.py").write_text(APP_PY, encoding="utf-8")
Path("requirements.txt").write_text("streamlit\nduckdb\npandas\nrequests\nanthropic\nopenai\ngoogle-genai\n", encoding="utf-8")

# Muestra reproducible de la tabla para publicar: el archivo completo es demasiado grande para GitHub
muestra = con.execute("SELECT * FROM retail USING SAMPLE reservoir(100000 ROWS) REPEATABLE (2026)").df()
RUTA_APP = "retail_app.duckdb"
if os.path.exists(RUTA_APP):
    os.remove(RUTA_APP)
con_app = duckdb.connect(RUTA_APP)
con_app.register("muestra", muestra)
con_app.execute("CREATE TABLE retail AS SELECT * FROM muestra")
con_app.close()
print("Archivos creados: app.py, requirements.txt y", RUTA_APP, f"({len(muestra):,} filas de muestra)")

Archivos creados: app.py, requirements.txt y retail_app.duckdb (100,000 filas de muestra)


## Cómo ejecutarla en Cursor (en su computador)
1. En Colab, descargue `app.py`, `requirements.txt` y `retail_app.duckdb` (panel de archivos de la izquierda) y guárdelos en una carpeta nueva; ábrala en Cursor.
2. Abra la terminal de Cursor y cree un entorno de Python: `python -m venv .venv`, actívelo e instale con `pip install -r requirements.txt`.
3. Defina el proveedor y la clave **como variables de entorno** (no en el código): `LLM_PROVEEDOR` (`ollama`, `anthropic`, `openai` o `gemini`), `LLM_MODELO` y, si usa API, `LLM_API_KEY`. Con Ollama no hace falta clave: instálelo, descargue un modelo pequeño y deje `LLM_PROVEEDOR=ollama`.
4. Ejecute `streamlit run app.py`. Se abre una ventana de chat en el navegador.
5. Pruebe con tres preguntas: una que el asistente deba responder bien, otra que deba bloquearse (por ejemplo, pedir que borre registros) y otra ambigua. Anote lo que ocurre.

## Cómo publicarla (opcional)
1. Suba a un repositorio de GitHub `app.py`, `requirements.txt` y `retail_app.duckdb`. **No suba claves ni archivos `.env`.**
2. En Streamlit Community Cloud, cree una aplicación desde ese repositorio y escriba los secretos en *Settings → Secrets* (por ejemplo `LLM_PROVEEDOR`, `LLM_MODELO`, `LLM_API_KEY` y `APP_PASSWORD`). Verifique las condiciones vigentes de la plataforma y de su proveedor de modelo.
3. Recuerde que la aplicación publicada usa una **muestra** de 100 000 filas: los resultados numéricos pueden diferir de los de la tabla completa de este notebook.

## Pregunta 11 (opcional)
**Respuesta:** En el notebook y en la aplicación se mantienen el validador, el acceso de solo lectura y el límite de filas. Al publicar, la clave debe quedar en secretos del servidor, nunca en el código ni en el navegador. Una aplicación accesible sin autenticación puede recibir consultas automatizadas y consumir la cuota de API; hacen falta autenticación, límites por usuario y monitoreo.

# 8. Actividad individual — Semana 14
Cuestionario individual de 10 ítems de selección múltiple sobre los contenidos de la semana y los resultados de este laboratorio. Conserve los resultados del notebook: varias preguntas pueden exigir interpretar su evaluación.

# 9. Informe ejecutivo
Máximo **12 líneas**. Incluya: número de preguntas evaluadas, tasa de acierto, tasa de alucinación, tasa de error semántico, un ejemplo de falla, dos mitigaciones, una limitación, la condición mínima para uso empresarial y la recomendación final: **usar, usar con supervisión o no usar todavía**.

In [42]:
# Informe ejecutivo actualizado con los resultados actuales (máximo 12 líneas)
informe = """Se evaluaron 10 preguntas de Online Retail II; 9 coincidieron con las consultas de referencia (90%).
La tasa de alucinación de esquema fue 0%; no hubo errores de sintaxis/ejecución, bloqueos ni errores de formato.
Se observó 1 error semántico (10%), correspondiente a la pregunta 7.
Ejemplo: la consulta del mes de 2011 con mayores ventas se ejecutó, pero el resultado difirió de la referencia.
Mitigación 1: precisar el periodo 2011, el agrupamiento por año-mes y la métrica SUM(sales).
Mitigación 2: repetir la batería y comparar cada resultado con su consulta de referencia antes de usarlo.
La conexión de solo lectura y el límite de filas ayudan; una lista de palabras prohibidas por regex no es infalible.
Limitación: la batería contiene solo 10 preguntas predefinidas y no cubre todas las consultas ni usuarios.
Condición mínima empresarial: ampliar pruebas representativas, conservar revisión humana, controlar accesos y monitorear errores.
Recomendación: usar con supervisión; revisar la falla semántica y volver a medir antes de automatizar decisiones."""
print(informe)
print("Líneas:", len([linea for linea in informe.strip().splitlines() if linea.strip()]))

Se evaluaron 10 preguntas de Online Retail II; 9 coincidieron con las consultas de referencia (90%).
La tasa de alucinación de esquema fue 0%; no hubo errores de sintaxis/ejecución, bloqueos ni errores de formato.
Se observó 1 error semántico (10%), correspondiente a la pregunta 7.
Ejemplo: la consulta del mes de 2011 con mayores ventas se ejecutó, pero el resultado difirió de la referencia.
Mitigación 1: precisar el periodo 2011, el agrupamiento por año-mes y la métrica SUM(sales).
Mitigación 2: repetir la batería y comparar cada resultado con su consulta de referencia antes de usarlo.
La conexión de solo lectura y el límite de filas ayudan; una lista de palabras prohibidas por regex no es infalible.
Limitación: la batería contiene solo 10 preguntas predefinidas y no cubre todas las consultas ni usuarios.
Condición mínima empresarial: ampliar pruebas representativas, conservar revisión humana, controlar accesos y monitorear errores.
Recomendación: usar con supervisión; revisar la fall

## 10. Ticket de salida
El asistente coincidió con 9 de 10 consultas de referencia (90%) y no presentó alucinaciones de esquema. La única falla fue semántica: la pregunta 7 se ejecutó, pero el mes de mayores ventas de 2011 difirió de la referencia. Recomiendo usarlo con supervisión, precisar la métrica temporal y validar las consultas frente a referencias antes de automatizar decisiones.

## 11. Checklist de entrega
- [x] Fuente real de UCI 502, limpieza y tabla retail con resultados guardados.
- [x] Diccionario de datos y reglas de negocio explícitas.
- [x] Conexión a OpenAI configurada en generate_sql(); la clave se lee desde Secretos de Colab y no se imprime.
- [x] Validador probado: pasaron 8 casos y DuckDB rechazó una escritura en modo de solo lectura.
- [x] Batería guardada de 10 preguntas, comparación con referencias e indicadores interpretados.
- [x] Preguntas de análisis, mitigaciones, informe ejecutivo de 10 líneas y ticket de salida completos.
- [ ] Copia del notebook D14 en semana14/ del repositorio personal: pendiente de sincronizar a Git.
- [ ] Cuestionario individual de Canvas: se responde por separado.
- [ ] Prueba opcional de la aplicación Streamlit con tres consultas.

# Referencias
- Chen, D. (2012). *Online Retail II* [Conjunto de datos]. UCI Machine Learning Repository, ID 502.
- DuckDB. Documentación oficial: https://duckdb.org/docs/
- Davenport, T. H. y Mittal, N. *All-in on AI*.
- Sílabo Analítica Empresarial Integrada — TECSUP 2026-II.